In [1]:
import os
import shutil
import random
import yaml
from pathlib import Path
from collections import OrderedDict

In [2]:
BASE = Path(r"D:\DELL\Desktop\Drones_training_results")
MERGED = BASE / "merged_dataset"
SAMPLES_PER_DATASET = 5000
SPLIT_RATIOS = (0.8, 0.1, 0.1)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

SOURCES = OrderedDict([
    ("uav300", {
        "path": BASE / "Drone_Datasets" / "Full_Dataset" / "AntiUAV_YOLO",
        "image_ext": ".jpg",
    }),
    ("visiodect", {
        "path": BASE / "Drone_Datasets" / "Full_Dataset" / "VisioDECT_YOLO_GroupSplit",
        "image_ext": ".jpg",
    }),
    ("dronedet21", {
        "path": BASE / "DroneDetect2021-train-results YOLO11M-MODEL" / "Drones",
        "image_ext": ".jpg",
    }),
])

MERGED_CLASSES = {
    0: "drone"
}

In [3]:
def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)


def normalize_label_to_class0(src_path, dst_path):
    with open(src_path, 'r') as f:
        lines = f.readlines()
    with open(dst_path, 'w') as f:
        for line in lines:
            parts = line.strip().split()
            if parts:
                parts[0] = '0'
                line = ' '.join(parts) + '\n'
            f.write(line)


def count_files(directory, ext):
    return len(list(directory.glob(f"*{ext}")))


def hardlink_or_copy(src, dst):
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

In [4]:
def collect_pairs(ds_path, ext):
    pairs = []
    for split in ["train", "val", "test"]:
        img_dir = ds_path / "images" / split
        lbl_dir = ds_path / "labels" / split
        if not img_dir.exists() or not lbl_dir.exists():
            continue
        for img_file in sorted(img_dir.glob(f"*{ext}")):
            lbl_file = lbl_dir / f"{img_file.stem}.txt"
            if lbl_file.exists():
                pairs.append((img_file, lbl_file))
    return pairs


def merge_splits(samples_per_dataset=SAMPLES_PER_DATASET, ratios=SPLIT_RATIOS):
    src_img_train = MERGED / "images" / "train"
    src_img_val = MERGED / "images" / "val"
    src_img_test = MERGED / "images" / "test"
    src_lbl_train = MERGED / "labels" / "train"
    src_lbl_val = MERGED / "labels" / "val"
    src_lbl_test = MERGED / "labels" / "test"
    ensure_dir(src_img_train)
    ensure_dir(src_img_val)
    ensure_dir(src_img_test)
    ensure_dir(src_lbl_train)
    ensure_dir(src_lbl_val)
    ensure_dir(src_lbl_test)

    total_train = 0
    total_val = 0
    total_test = 0

    for ds_name, ds_info in SOURCES.items():
        ds_path = ds_info["path"]
        ext = ds_info["image_ext"]

        all_pairs = collect_pairs(ds_path, ext)
        if not all_pairs:
            print(f"  WARNING: {ds_name} has no image/label pairs")
            continue

        n_sample = min(samples_per_dataset, len(all_pairs))
        sampled = random.sample(all_pairs, n_sample)
        random.shuffle(sampled)

        n_total = len(sampled)
        n_train = round(n_total * ratios[0] / sum(ratios))
        n_val = round(n_total * ratios[1] / sum(ratios))
        n_test = n_total - n_train - n_val

        splits = [
            (sampled[:n_train], src_img_train, src_lbl_train),
            (sampled[n_train:n_train + n_val], src_img_val, src_lbl_val),
            (sampled[n_train + n_val:], src_img_test, src_lbl_test),
        ]

        ds_img_count = 0
        for pairs, img_dir, lbl_dir in splits:
            for img_file, lbl_file in pairs:
                new_img_name = f"{ds_name}_{img_file.name}"
                new_lbl_name = f"{ds_name}_{lbl_file.name}"
                hardlink_or_copy(str(img_file), str(img_dir / new_img_name))
                normalize_label_to_class0(str(lbl_file), str(lbl_dir / new_lbl_name))
                ds_img_count += 1

        print(f"  {ds_name}: {n_train} train, {n_val} val, {n_test} test ({ds_img_count} total)")
        total_train += n_train
        total_val += n_val
        total_test += n_test

    return total_train, total_val, total_test

In [5]:
def write_merged_data_yaml():
    data = {
        "path": str(MERGED.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(MERGED_CLASSES),
        "names": list(MERGED_CLASSES.values()),
    }
    yaml_path = MERGED / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False)
    print(f"Written: {yaml_path}")

In [6]:
print("=" * 60)
print("Building Randomly Sampled Merged Dataset")
print(f"  Samples per dataset: {SAMPLES_PER_DATASET}")
print(f"  Split ratios (train/val/test): {SPLIT_RATIOS}")
print("=" * 60)

if MERGED.exists():
    print(f"Removing existing merged dataset: {MERGED}")
    shutil.rmtree(MERGED)
ensure_dir(MERGED)

print("\n--- Randomly Sampling and Splitting Datasets ---")
train_img, val_img, test_img = merge_splits()
print(f"\n  Merged train: {train_img} images")
print(f"  Merged val:   {val_img} images")
print(f"  Merged test:  {test_img} images")

print("\n--- Writing YAML Config ---")
write_merged_data_yaml()

print("\n" + "=" * 60)
print("Merged dataset created successfully!")
print(f"Location: {MERGED}")
print("=" * 60)

Building Randomly Sampled Merged Dataset
  Samples per dataset: 5000
  Split ratios (train/val/test): (0.8, 0.1, 0.1)

--- Randomly Sampling and Splitting Datasets ---
  uav300: 4000 train, 500 val, 500 test (5000 total)
  visiodect: 3302 train, 413 val, 413 test (4128 total)
  dronedet21: 4000 train, 500 val, 500 test (5000 total)

  Merged train: 11302 images
  Merged val:   1413 images
  Merged test:  1413 images

--- Writing YAML Config ---
Written: D:\DELL\Desktop\Drones_training_results\merged_dataset\data.yaml

Merged dataset created successfully!
Location: D:\DELL\Desktop\Drones_training_results\merged_dataset
